# Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
import plotly.io as pio
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.metrics import root_mean_squared_error as rmse
import seaborn as sns
from brokenaxes import brokenaxes
#----------------------------------------
import warnings
warnings.filterwarnings('ignore')

# 1) Functions

In [2]:
def load_datasets(path1, path2, path3):

    df = pd.read_csv(path1, sep = ";")
    df['year'] = pd.to_datetime(df['year'], format = '%d/%m/%Y')
    #-----------------------------------------------------------------------------------------
    df1 = pd.read_csv(path2, sep = ";")
    df1['date'] = pd.to_datetime(df1['date'], dayfirst = True, format = 'mixed')
    df1['month'] = df1['date'].dt.month_name()
    df1 = df1.fillna('nan')
    #-----------------------------------------------------------------------------------------
    df2 = pd.read_csv(path3, sep = ";")
    df2['date_start'] = pd.to_datetime(df2['date_start'], dayfirst = False, format = 'mixed')
    df2['date_end'] = pd.to_datetime(df2['date_end'], dayfirst = False, format = 'mixed')
    df2['days'] = (df2['date_end'] - df2['date_start']).dt.days
    p1 = df2.loc[df2['site'] == 'P1', :].reset_index(drop=True)
    p2 = df2.loc[df2['site'] == 'P2', :].reset_index(drop=True)
    p3 = df2.loc[df2['site'] == 'P3', :].reset_index(drop=True)
    
    return df, df1, p1, p2, p3
#------------------------------------------------------------------------------------
def land_use_land_cover(dataframe):

    fig1, axs = plt.subplots(figsize = (8,6))
    fig1.tight_layout(pad = 3)
    fig1.suptitle('Temporal series of resident population and land use/land cover', fontsize = 13)
    axs.plot(dataframe['year'], dataframe['urban'], color = 'red')
    axs.tick_params(axis='y', labelcolor= 'red')
    axs.set_ylabel('Urban area (ha)', color = 'red', fontsize = 12, labelpad = 10)
    axs.set_yticks([0, 250, 500, 750, 1000, 1250, 1500, 1750, 2000, 2250, 2500])
    
    ax2 = axs.twinx()

    ax2.plot(df['year'], df['population'], color = 'black')
    ax2.set_ylabel('Resident population', color = 'black', rotation = -90, fontsize = 12, labelpad = 15)
    ax2.tick_params(axis='y', labelcolor= 'black')
    ax2.spines['right'].set_position(('outward', 0))
    ax2.set_yticks([0, 20000, 40000, 60000, 80000, 100000, 120000, 140000, 160000, 180000, 200000])

    ax3 = axs.twinx()

    ax3.plot(df['year'], df['forest'], color = 'green')
    ax3.set_ylabel('Forest (ha)', color = 'green', fontsize = 12, labelpad = 10)
    ax3.tick_params(axis='y', labelcolor= 'green')
    ax3.spines['right'].set_position(('axes', -0.25))
    ax3.set_yticks([0, 2600, 5200, 7800, 10400, 13000, 15600, 18200, 20800, 23400, 26000])
    ax3.yaxis.set_label_coords(-0.3, 0.5)

    ax4 = axs.twinx()

    ax4.plot(df['year'], df['agriculture'], color = 'orange')
    ax4.set_ylabel('Agriculture (ha)', color = 'orange', fontsize = 12, labelpad = 10)
    ax4.tick_params(axis='y', labelcolor= 'orange')
    ax4.spines['right'].set_position(('axes', -0.44))
    ax4.set_yticks([0, 10000, 20000, 30000, 40000, 50000, 60000, 70000, 80000, 90000, 100000])
    ax4.yaxis.set_label_coords(-0.49, 0.5)

    ax5 = axs.twinx()
    
    ax5.plot(df['year'], df['water'], color = 'blue')
    ax5.set_ylabel('Water (ha)', color = 'blue', fontsize = 12, labelpad = 10)
    ax5.tick_params(axis='y', labelcolor= 'blue')
    ax5.spines['right'].set_position(('axes', -0.63))
    ax5.set_yticks([0, 300, 600, 900, 1200, 1500, 1800, 2100, 2400, 2700, 3000])
    ax5.yaxis.set_label_coords(-0.68, 0.5)

    plt.savefig('../figures/land_use_land_cover.tiff', bbox_inches='tight')
    plt.close(fig1)
#------------------------------------------------------------------------------------
def energy_effluent_time_series(dataframe):

    dataframe = dataframe.loc[(dataframe['date'] >= pd.to_datetime('2016-01-01'))&
                              (dataframe['date'] < pd.to_datetime('2023-01-01')), :]
    dataframe = dataframe.set_index('date')
   
    fig2, ax = plt.subplots(figsize=(12,5))

    ax.set_xlabel('Month/Year')

    ax.plot(dataframe.index, dataframe['effluent_outflow'], color = 'black')
    ax.set_xlabel('Month/Year', fontsize = 12)
    ax.set_ylabel('Effluent outflow (m3/month)', fontsize = 12, labelpad = 10, color = 'black')
    ax.tick_params(axis = 'x', rotation = 90, labelsize = 9)
    ax.set_ylim([68000, 86000])
    ax.set_yticks([68000, 71000, 74000, 77000, 80000, 83000, 86000])

    ax2 = ax.twinx()

    ax2.plot(dataframe.index, dataframe['monthly_electricity_consumption'], color = 'blue', alpha = 0.7)
    ax2.set_ylabel('Energy consumption (MWh)', color = 'blue', fontsize = 12, rotation = -90, labelpad = 20)
    ax2.set_ylim([700, 3700])
    ax2.set_yticks([700, 1200, 1700, 2200, 2700, 3200, 3700])
    ax2.tick_params(axis='x', labelrotation = 90, labelsize = 9)

    plt.savefig('../figures/time_series.tiff', bbox_inches='tight')
    plt.close(fig2)
#------------------------------------------------------------------------------------
def energy_effluent_time_series2(dataframe):

    dataframe = dataframe.loc[(dataframe['date'] >= pd.to_datetime('2016-01-01'))&
                              (dataframe['date'] < pd.to_datetime('2023-01-01')), :]

    fig2b, ax = plt.subplots(figsize=(12,5), dpi=400)
    plt.gca().axes.get_xaxis().set_visible(False)
    plt.gca().axes.get_yaxis().set_visible(False)

    bax = brokenaxes(
                     ylims=((0, 1), (68000, 84000)),
                     height_ratios=[100, 1],
                     hspace=0.1,
                     #wspace=0.2,#the width of x axis broken             
                     despine=True,
                     diag_color='r')

    bax.plot(dataframe['date'].dt.strftime("%Y-%m"), dataframe['effluent_outflow'], color = 'black', marker = '*', markersize = 5)
    bax.set_xlabel('Month/Year', fontsize = 12, labelpad = 60)
    bax.set_ylabel('Effluent outflow (m3/month)', fontsize = 12, labelpad = 50, color = 'black')
    bax.tick_params(axis = 'x', rotation = 90, labelsize = 10)

    ax2 = ax.twinx()

    ax2.plot(dataframe['date'], dataframe['monthly_electricity_consumption'], color = 'blue', marker = 'o', markersize = 3, alpha = 0.7)
    ax2.set_ylabel('Energy consumption (MWh)', color = 'blue', fontsize = 12, rotation = -90, labelpad = 20)
    ax2.set_ylim([0, 3500]);
   
    plt.savefig('../figures/time_series2.tiff', bbox_inches='tight')
    plt.close(fig2b)
#------------------------------------------------------------------------------------
def monthly_groups(dataframe):

    dataframe = dataframe.loc[(dataframe['date'] >= pd.to_datetime('2016-01-01'))&
                              (dataframe['date'] < pd.to_datetime('2023-01-01')), :]

    dataframe[['monthly_electricity_consumption', 'effluent_outflow', 'resident_population']] = dataframe[['monthly_electricity_consumption', 'effluent_outflow', 'resident_population']].convert_dtypes()
    
    dataframe = dataframe.set_index('date')
    months = list(dataframe['month'].unique())
    #-------------------------------------------------------------------------
    q1_electricity = dataframe[['month', 'monthly_electricity_consumption']].groupby('month').quantile(0.25).reset_index()
    q2_electricity = dataframe[['month', 'monthly_electricity_consumption']].groupby('month').quantile(0.50).reset_index()
    q3_electricity = dataframe[['month', 'monthly_electricity_consumption']].groupby('month').quantile(0.75).reset_index()                   
    #-------------------------------------------------------------------------
    q1_effluent = dataframe[['month', 'effluent_outflow']].groupby('month').quantile(0.25).reset_index()
    q2_effluent = dataframe[['month', 'effluent_outflow']].groupby('month').quantile(0.50).reset_index()
    q3_effluent = dataframe[['month', 'effluent_outflow']].groupby('month').quantile(0.75).reset_index()                   
    #-------------------------------------------------------------------------
    electricity = q1_electricity.merge(q2_electricity, on = 'month', how = 'inner')
    electricity = electricity.merge(q3_electricity, on = 'month', how = 'inner')

    electricity['month'] = pd.Categorical(electricity['month'], categories = months)
    electricity = electricity.sort_values('month')
    electricity['season'] = electricity['month'].apply(lambda x: 'Dry' if (x == 'January')|
                                                                          (x == 'February')|
                                                                          (x == 'September')|
                                                                          (x == 'October')|
                                                                          (x == 'November')|
                                                                          (x == 'December')
                                                                       else 'Wet'
                                                      )

    electricity.columns = ['Month', 'Quantile (25%)', 'Median', 'Quantile (75%)', 'Season']
    electricity['error_inf'] = electricity['Median'] - electricity['Quantile (25%)']
    electricity['error_sup'] = electricity['Quantile (75%)'] - electricity['Median']
    #-------------------------------------------------------------------------
    effluent = q1_effluent.merge(q2_effluent, on = 'month', how = 'inner')
    effluent = effluent.merge(q3_effluent, on = 'month', how = 'inner')

    effluent['month'] = pd.Categorical(effluent['month'], categories = months)
    effluent = effluent.sort_values('month')
    effluent['season'] = effluent['month'].apply(lambda x: 'Dry' if (x == 'January')|
                                                                    (x == 'February')|
                                                                    (x == 'September')|
                                                                    (x == 'October')|
                                                                    (x == 'November')|
                                                                    (x == 'December')
                                                                 else 'Wet'
                                                )

    effluent.columns = ['Month', 'Quantile (25%)', 'Median', 'Quantile (75%)', 'Season']
    effluent['error_inf'] = effluent['Median'] - effluent['Quantile (25%)']
    effluent['error_sup'] = effluent['Quantile (75%)'] - effluent['Median']
    
    electricity.to_excel("../dataframes/electricity.xlsx", index=False)
    effluent.to_excel("../dataframes/effluent.xlsx", index=False)
    #-------------------------------------------------------------------------------------------------------
    rainfall_groups = dataframe[(dataframe['monthly_rainfall'] != 'nan')][['month', 'monthly_rainfall']].groupby('month').median().reset_index()
    rainfall_groups['month'] = pd.Categorical(rainfall_groups['month'], categories = months)
    rainfall_groups = rainfall_groups.sort_values('month')
    #-------------------------------------------------------------------------------------------------------
    fig3, ax3 = plt.subplots(figsize = (5, 4))
    plt.gca().axes.get_xaxis().set_visible(False)
    plt.gca().axes.get_yaxis().set_visible(False)
    
    bax = brokenaxes(ylims=((0, 1), (68000, 84000)),
                     height_ratios=[100, 1],
                     hspace=0.1,
                     diag_color='black',
                     despine=True)

    erro_inf = list(effluent['error_inf'])
    erro_sup = list(effluent['error_sup'])
    erros_customizados = np.array([erro_inf, erro_sup])

    bax.bar(effluent['Month'], 
            effluent['Median'], 
            color = ['g', 'g', 'r', 'r', 'r', 'r', 'r', 'r', 'g', 'g', 'g', 'g'],
            yerr = erros_customizados, capsize=2, ecolor='blue', error_kw = {'ls': 'dashed'})

    bax.axs[0].set_yticks([68000, 72000, 76000, 80000, 84000])

    bax.tick_params(axis= 'x', labelrotation = 90)
    bax.legend(frameon = False)
    bax.set_xlabel('Month', fontsize = 11, labelpad = 50)
    bax.set_ylabel('Median Effluent Outflow (m3/month)', fontsize = 11, labelpad = 50)

    ax2 = ax3.twinx()

    ax2.plot(rainfall_groups['month'], rainfall_groups['monthly_rainfall'], marker = '*', color = 'black')
    ax2.set_ylabel('Monthly rainfall', fontsize = 11, rotation = -90, labelpad = 20)

    ax2.set_ylim([0,350])
 
    #sns.barplot(effluent, 
                #x = 'Month', 
                #y = 'Median', 
                #hue = 'Season',
                #palette = ['green', 'red'],
                #width = .8, legend = True, 
                #ax=ax3)

    #ax3.tick_params(axis= 'x', labelrotation = 90)
    #ax3.legend(frameon = False)
    #ax3.set_xlabel('Month', fontsize = 11)
    #ax3.set_ylabel('Median Effluent Outflow (m3/month)', fontsize = 11, labelpad = 10)
    #ax3.set_ylim(68000,88000)
    #ax3.set_yticks([68000, 72000, 76000, 80000, 84000, 88000]);

    plt.savefig('../figures/monthly_effluent_rainfall.tiff', bbox_inches='tight')
    plt.close(fig3)
    #-------------------------------------------------------------------------------------------------------
    fig4, ax4 = plt.subplots(figsize = (5, 4))

    sns.barplot(electricity, 
                x = 'Month', 
                y = 'Median', 
                hue = 'Season',
                palette = ['green', 'red'],
                width = .8, legend = True, 
                ax=ax4)

    ax4.tick_params(axis= 'x', labelrotation = 90)
    ax4.legend(frameon = False)
    ax4.set_xlabel('Month', fontsize = 11)
    ax4.set_ylabel('Median Energy Consumption (MWh)', fontsize = 11, labelpad = 10);
    ax4.set_ylim(0,3500)
    ax4.set_yticks([0, 700, 1400, 2100, 2800, 3500]);

    plt.savefig('../figures/energy_monthly_groups.tiff', bbox_inches='tight')
    plt.close(fig4)

    return electricity, effluent, rainfall_groups
#------------------------------------------------------------------------------------
def elemental_contents(p1, p2, p3):

    P1 = [
         ["P1", "P1", "P1", 'P1', "P1", "P1", "P1", 'P1', "P1", "P1", "P1", 'P1'],
         ['Sep-Oct/22', 'Oct-Nov/22', 'Nov-Dec/22', 'Dec-Jan/23', 'Jan-Feb/23', 'Feb-Mar/23', 'Mar-Apr/23', 'Apr-May/23', 'May-Jun/23', 'Jun-Jul/23', 'Jul-Aug/23', 'Aug-Sep/23']
         ]
    
    P2 = [
         ["P2", "P2", "P2", 'P2', "P2", "P2", "P2", 'P2', "P2", "P2", "P2", 'P2'],
         ['Sep-Oct/22', 'Oct-Nov/22', 'Nov-Dec/22', 'Dec-Jan/23', 'Jan-Feb/23', 'Feb-Mar/23', 'Mar-Apr/23', 'Apr-May/23', 'May-Jun/23', 'Jun-Jul/23', 'Jul-Aug/23', 'Aug-Sep/23']
         ]
    
    P3 = [
         ["P3", "P3", "P3", "P3", "P3", "P3", "P3", "P3", "P3", "P3", "P3", "P3"],
         ['Sep-Oct/22', 'Oct-Nov/22', 'Nov-Dec/22', 'Dec-Jan/23', 'Jan-Feb/23', 'Feb-Mar/23', 'Mar-Apr/23', 'Apr-May/23', 'May-Jun/23', 'Jun-Jul/23', 'Jul-Aug/23', 'Aug-Sep/23']
         ]
    #--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
    fig5 = make_subplots(rows=3, cols=2, vertical_spacing=0.25, horizontal_spacing=0.1)
    fig5.update_yaxes(linecolor='black')
    fig5.update_xaxes(linecolor='black')
    fig5.update_yaxes(ticks="outside")
    
    fig5.add_trace(go.Bar(x = P1, y = p1['oc'], name = 'OC', marker_color = 'gray', orientation = 'v'), row = 1, col = 1)
    fig5.add_trace(go.Bar(x = P1, y = p1['tn'], name = 'TN', marker_color = 'red', orientation = 'v'), row = 1, col = 1)
    fig5.update_layout(barmode = 'stack', plot_bgcolor = 'white')
    fig5.update_yaxes(title_text = "Contents (%)", range=[0, 3], row=1, col=1)
    fig5.update_traces(marker_line_color='rgb(0,0,0)', marker_line_width=1, row = 1, col = 1)
    #--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
    fig5.add_trace(go.Bar(x = P2, y = p2['oc'], marker_color = 'gray', orientation = 'v', showlegend = False), row = 2, col = 1)
    fig5.add_trace(go.Bar(x = P2, y = p2['tn'], marker_color = 'red', orientation = 'v', showlegend = False), row = 2, col = 1)
    fig5.update_layout(barmode = 'stack', plot_bgcolor = 'white')
    fig5.update_yaxes(title_text = "Contents (%)", range=[0, 3], row=2, col=1)
    fig5.update_traces(marker_line_color='rgb(0,0,0)', marker_line_width=1, row = 2, col = 1)
    #--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
    fig5.add_trace(go.Bar(x = P3, y = p3['oc'], marker_color = 'gray', orientation = 'v', showlegend = False), row = 3, col = 1)
    fig5.add_trace(go.Bar(x = P3, y = p3['tn'], marker_color = 'red', orientation = 'v', showlegend = False), row = 3, col = 1)
    fig5.update_layout(barmode = 'stack', plot_bgcolor = 'white')
    fig5.update_yaxes(title_text = "Contents (%)", range=[0, 3], row=3, col=1)
    fig5.update_traces(marker_line_color='rgb(0,0,0)', marker_line_width=1, row = 3, col = 1)
    #--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
    fig5.add_trace(go.Scatter(x = P1, y = p1['CN_ratio'], name = 'C/N ratio', mode='lines+markers', line_color='#000000', showlegend = False), row = 1, col = 2)
    fig5.update_yaxes(range=[6, 10], row=1, col=2)
    
    fig5.add_trace(go.Scatter(x = P2, y = p2['CN_ratio'], name = 'C/N ratio', mode='lines+markers', line_color='#000000', showlegend = False), row = 2, col = 2)
    fig5.update_yaxes(range=[6, 10], row=2, col=2)
    
    fig5.add_trace(go.Scatter(x = P3, y = p3['CN_ratio'], name = 'C/N ratio', mode='lines+markers', line_color='#000000', showlegend = True), row = 3, col = 2)
    fig5.update_yaxes(range=[6, 10], row=3, col=2)
    #--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
    fig5.update_layout(
        width=1000,
        height=800,
        title_text="Temporal variation of elemental contents and (C/N) ratio"
    )

    return fig5
#------------------------------------------------------------------------------------
def linear_regression(p1, p2, p3):

    model_ = LinearRegression(fit_intercept = True)
    model_.fit(pd.DataFrame(pd.concat([p1, p2, p3])['oc']), 
               pd.DataFrame(pd.concat([p1, p2, p3])['tn']) 
               )
    tn_predict = model_.predict(pd.DataFrame(pd.concat([p1, p2, p3])['oc']))
    r2_score_ = round(r2_score(pd.DataFrame(pd.concat([p1, p2, p3])['tn']), tn_predict),3)
    slope = round(list(model_.coef_[0])[0],3) 
    intercept = round(list(model_.intercept_)[0],4)
    #------------------------------------------------------------------------------------
    scatter = px.scatter(pd.concat([p1, p2, p3]), 
                     x = pd.concat([p1, p2, p3])['oc'], 
                     y = pd.concat([p1, p2, p3])['tn'],
                     color = pd.concat([p1, p2, p3])['site'],
                     trendline="ols",
                     trendline_scope="overall")

    scatter.update_traces(line=dict(width=1, dash='solid'), line_color = 'black', selector=dict(mode='lines'))

    fig6 = make_subplots(rows=1, cols=1)

    fig6.update_layout(width = 600,
                       height = 500,
                       title_text="Cross-plot of total nitrogen (TN) versus organic carbon (OC)", 
                       legend_title_text = 'Sites')
    fig6.update_yaxes(linecolor='black')
    fig6.update_xaxes(linecolor='black')
    fig6.update_yaxes(ticks="outside")
    fig6.update_xaxes(ticks="outside")

    fig6.update_layout(plot_bgcolor = 'white')
    fig6.update_yaxes(title_text = "TN (%)", range=[0, 0.5], row=1, col=1)
    fig6.update_xaxes(title_text = "OC (%)", range=[0, 2.5], row=1, col=1)

    for trace in scatter.data:
        fig6.add_trace(trace, row=1, col=1)

    fig6.add_annotation(x=0.9, y=0.4,
                text="TN (%) = {} * OC (%) + {}<br><i>R</i><sup>2</sup>: {}".format(slope, intercept, r2_score_),
                showarrow=False,
                yshift=18)

    return fig6
#------------------------------------------------------------------------------------
def isotope_ratios(p1, p2, p3):

    P1 = [
         ["P1", "P1", "P1", 'P1', "P1", "P1", "P1", 'P1', "P1", "P1", "P1", 'P1'],
         ['Sep-Oct/22', 'Oct-Nov/22', 'Nov-Dec/22', 'Dec-Jan/23', 'Jan-Feb/23', 'Feb-Mar/23', 'Mar-Apr/23', 'Apr-May/23', 'May-Jun/23', 'Jun-Jul/23', 'Jul-Aug/23', 'Aug-Sep/23']
         ]
    
    P2 = [
             ["P2", "P2", "P2", 'P2', "P2", "P2", "P2", 'P2', "P2", "P2", "P2", 'P2'],
             ['Sep-Oct/22', 'Oct-Nov/22', 'Nov-Dec/22', 'Dec-Jan/23', 'Jan-Feb/23', 'Feb-Mar/23', 'Mar-Apr/23', 'Apr-May/23', 'May-Jun/23', 'Jun-Jul/23', 'Jul-Aug/23', 'Aug-Sep/23']
             ]
        
    P3 = [
             ["P3", "P3", "P3", "P3", "P3", "P3", "P3", "P3", "P3", "P3", "P3", "P3"],
             ['Sep-Oct/22', 'Oct-Nov/22', 'Nov-Dec/22', 'Dec-Jan/23', 'Jan-Feb/23', 'Feb-Mar/23', 'Mar-Apr/23', 'Apr-May/23', 'May-Jun/23', 'Jun-Jul/23', 'Jul-Aug/23', 'Aug-Sep/23']
             ]
    #--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
    fig7 = make_subplots(rows=3, cols=2, vertical_spacing=0.25, horizontal_spacing=0.1)
    fig7.update_yaxes(linecolor='black')
    fig7.update_xaxes(linecolor='black')
    fig7.update_yaxes(ticks="outside")
        
    fig7.add_trace(go.Scatter(x = P1, y = p1['delta_13C'], mode='lines+markers', line_color='#000000', showlegend = False), row = 1, col = 1)
    fig7.update_layout(plot_bgcolor = 'white')
    fig7.update_yaxes(title_text = "<i>\u03b4</i><sup>13</sup>C (‰)", range=[-22, -16], row=1, col=1)
    fig7.update_traces(marker_line_color='rgb(0,0,0)', marker_line_width=1, row = 1, col = 1)
    #--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
    fig7.add_trace(go.Scatter(x = P2, y = p2['delta_13C'], mode='lines+markers', line_color='#000000', showlegend = False), row = 2, col = 1)
    fig7.update_layout(plot_bgcolor = 'white')
    fig7.update_yaxes(title_text = "<i>\u03b4</i><sup>13</sup>C (‰)", range=[-22, -16], row=2, col=1)
    fig7.update_traces(marker_line_color='rgb(0,0,0)', marker_line_width=1, row = 2, col = 1)
    #--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
    fig7.add_trace(go.Scatter(x = P3, y = p3['delta_13C'], mode='lines+markers', line_color='#000000', showlegend = False), row = 3, col = 1)
    fig7.update_layout(plot_bgcolor = 'white')
    fig7.update_yaxes(title_text = "<i>\u03b4</i><sup>13</sup>C (‰)", range=[-22, -16], row=3, col=1)
    fig7.update_traces(marker_line_color='rgb(0,0,0)', marker_line_width=1, row = 3, col = 1)
    #--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
    fig7.add_trace(go.Scatter(x = P1, y = p1['delta_15N'], mode='lines+markers', line_color='#000000', showlegend = False), row = 1, col = 2)
    fig7.update_yaxes(title_text = "<i>\u03b4</i><sup>15</sup>N (‰)", range=[1, 8], row=1, col=2)
        
    fig7.add_trace(go.Scatter(x = P2, y = p2['delta_15N'], mode='lines+markers', line_color='#000000', showlegend = False), row = 2, col = 2)
    fig7.update_yaxes(title_text = "<i>\u03b4</i><sup>15</sup>N (‰)", range=[1, 8], row=2, col=2)
        
    fig7.add_trace(go.Scatter(x = P3, y = p3['delta_15N'], mode='lines+markers', line_color='#000000', showlegend = True), row = 3, col = 2)
    fig7.update_yaxes(title_text = "<i>\u03b4</i><sup>15</sup>N (‰)", range=[1, 8], row=3, col=2)
    #--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
    fig7.update_layout(
        width=1000,
        height=800,
        title_text="Temporal variation of stable carbon and nitrogen isotope ratios"
    )

    return fig7
#------------------------------------------------------------------------------------
def population_energy(dataframe):

    dataframe = dataframe.loc[(dataframe['resident_population'] != 'nan')&(dataframe['monthly_electricity_consumption'] != 'nan'), 
                              ['date', 'monthly_electricity_consumption', 'annual_electricity_consumption', 'resident_population']]
    
    dataframe = dataframe.convert_dtypes()
    dataframe.to_excel("../dataframes/population_energy.xlsx", index=False)
    #--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
    fig8, ax = plt.subplots(figsize = (6,5))
    fig8.tight_layout(pad = 4)

    sns.barplot(x = dataframe['date'].astype(str), y = dataframe['resident_population'], color = 'gray', ax = ax)
    ax.tick_params(axis='x', labelrotation = 90, labelsize = 9)
    ax.set_ylim([0, 30000])
    ax.set_yticks([0, 6000, 12000, 18000, 24000, 30000])
    ax.set_xlabel('Annual Census Date', fontsize = 12, labelpad = 10, color = 'black')
    ax.set_ylabel('Resident population', fontsize = 12, labelpad = 10, color = 'gray')

    ax2 = ax.twinx()

    sns.lineplot(x = dataframe['date'].astype(str), y = dataframe['monthly_electricity_consumption'], marker = 'o', color = 'r', ax = ax2)
    ax2.set_ylabel('Monthly Energy consumption (MWh)', color = 'red', fontsize = 12, rotation = -90, labelpad = 20)

    ax2.set_ylim([0, 2000])
    ax2.set_yticks([0, 400, 800, 1200, 1600, 2000])
    
    fig8.savefig('../figures/population_energy.tiff', bbox_inches='tight')
    plt.close(fig8);
#------------------------------------------------------------------------------------
def linear_regression_july(dataframe):

    #Month: July
    dataframe = dataframe.loc[(dataframe['resident_population'] != 'nan')&
                              (dataframe['monthly_electricity_consumption'] != 'nan'), 
                             ['date', 'resident_population', 'monthly_electricity_consumption']].reset_index(drop=True)
    
    dataframe = dataframe.convert_dtypes()

    model_ = LinearRegression(fit_intercept = True)
    model_.fit(dataframe[['monthly_electricity_consumption']], 
               dataframe[['resident_population']] 
            )

    population_predict = model_.predict(dataframe[['monthly_electricity_consumption']])
    dataframe['prediction'] = population_predict.round(0).astype(np.int64)

    slope = round(list(model_.coef_[0])[0],3)
    intercept = round(list(model_.intercept_)[0],2)
    r2_score_ = round(r2_score(dataframe['resident_population'], population_predict),2)
    rmse_july = round(rmse(dataframe['resident_population'], dataframe['prediction']),0)
    #-----------------------------------------------------------------------------
    fig9, ax = plt.subplots(figsize = (6,5))
    sns.regplot(data = dataframe,
                x = 'monthly_electricity_consumption',
                y = 'resident_population', color = 'black', 
                ax = ax)

    ax.set_xlabel('Electricity Consumption (MWh) for July')
    ax.set_ylabel('Resident Population')
    
    ax.set_ylim(15000,25000)
    ax.set_yticks([15000, 17500, 20000, 22500, 25000])
    
    ax.set_xlim(300,1800)
    ax.set_xticks([300, 600, 900, 1200, 1500, 1800])
    
    ax.annotate(xy = (350,24200), text = 'Population = {} * Energy + {}'.format(slope, intercept))
    ax.annotate(xy = (350,23500), text = 'R2 score: {}; RMSE: {} '.format(r2_score_, rmse_july))
    
    
    plt.savefig('../figures/regplot_energy_population.tiff', bbox_inches='tight')
    plt.close(fig9);
#------------------------------------------------------------------------------------

# 2) Loading datasets

In [3]:
path1 = '../datasets/land_use_cover.csv'
path2 = '../datasets/time_series_tamandare.csv'
path3 = '../datasets/elemental_isotopic_data.csv'

df = load_datasets(path1, path2, path3)[0]
df1 = load_datasets(path1, path2, path3)[1]
p1 = load_datasets(path1, path2, path3)[2]
p2 = load_datasets(path1, path2, path3)[3]
p3 = load_datasets(path1, path2, path3)[4]

In [4]:
df1.shape

(314, 8)

In [5]:
df1.columns

Index(['date', 'month', 'monthly_rainfall', 'monthly_electricity_consumption',
       'annual_electricity_consumption', 'effluent_outflow',
       'resident_population', 'year_census_ibge'],
      dtype='object')

In [6]:
df1.dtypes

date                               datetime64[ns]
month                                      object
monthly_rainfall                           object
monthly_electricity_consumption            object
annual_electricity_consumption             object
effluent_outflow                           object
resident_population                        object
year_census_ibge                           object
dtype: object

# 3) Figures and Dataframes

<h4 style='text-align: justify; color: black;'><u>Figure 1</u> - Time series of land use and land cover from 1985 to 2023 in the municipalities of Tamandaré, Barreiros, Sirinhaém, Rio Formoso, and São José da Coroa Grande. </h4>

In [ ]:
land_use_land_cover(df)

<h4 style='text-align: justify; color: black;'><u>Figure 2</u> -  Time series of monthly residential electricity consumption (MWh) (da Silveira and Ferreira, 2024) and effluent discharge from wastewater treatment plant of Tamandaré city (COMPESA, 2024). </h4>

In [ ]:
energy_effluent_time_series2(df1)

<h4 style='text-align: justify; color: black;'><u>Dataframe</u> - Dataframes of monthly median residential electricity consumption (MWh) (A) (da Silveira and Ferreira, 2024) and effluent discharge from wastewater treatment plant of Tamandaré city (B) (COMPESA, 2024).</h4>

In [ ]:
monthly_groups(df1)[0]

In [ ]:
monthly_groups(df1)[1]

In [ ]:
monthly_groups(df1)[2]

<h4 style='text-align: justify; color: black;'><u>Figure 3</u> - Time series of total nitrogen (TN), organic carbon (OC) and carbon-to-nitrogen stoichiometric ratio in trapped sediments collected at Tamandaré Reef Complex (sites P1, P2 and P3) between September 2022 and September 2023. </h4>

In [ ]:
elemental_contents(p1, p2, p3)

<h4 style='text-align: justify; color: black;'><u>Figure 4</u> - Cross-plot of total nitrogen (TN) and organic carbon (OC) in trapped sediments collected at Tamandaré Reef Complex (sites P1, P2 and P3) between September 2022 and September 2023.</h4>

In [ ]:
linear_regression(p1,p2,p3)

<h4 style='text-align: justify; color: black;'><u>Figure 5</u> - Time series of stable carbon and nitrogen isotope ratios in trapped sediments collected from Tamandaré Reef Complex (sites P1, P2 and P3) betwenn September 2022 and September 2023.</h4>

In [ ]:
isotope_ratios(p1, p2, p3)

<h4 style='text-align: justify; color: black;'><u>Figure 6</u> - Annual variation in resident population and residential electricity consumption in Tamandaré.</h4>

In [ ]:
population_energy(df1)

<h4 style='text-align: justify; color: black;'><u>Figure 7</u> - Linear regression model for predicting of resident population and tourist numbers in Tamandaré using residential electricity consumption in July (dataset from 1999 to 2021) (da Silveira and Ferreira, 2024).</h4>

In [ ]:
linear_regression_july(df1)

In [ ]:
figure, ax = plt.subplots(figsize = (6,5))
sns.regplot(data = dataframe2,
            x = 'monthly_electricity_consumption',
            y = 'resident_population', color = 'black', scatter = False,
            ax = ax)

ax.set_xlabel('Electricity Consumption (MWh)')
ax.set_ylabel('Resident population')

ax.set_ylim(15000,30000)
ax.set_yticks([15000, 18000, 21000, 24000, 27000, 30000])

ax.set_xlim(300,1800)
ax.set_xticks([300, 600, 900, 1200, 1500, 1800])

sns.scatterplot(data = dataframe14,
                x = 'monthly_electricity_consumption',
                y = 'pop_prediction_february', color = 'red', label = 'April', ax = ax)

ax.legend(frameon = False);